In [3]:
#apply StandardScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch
from sklearn.decomposition import PCA

#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

# Drop target and ID column & target column
X = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape:", X.shape)

Features shape: (96000, 7)


In [4]:
#Apply StandardScaler
scaler = StandardScaler()
X_sp = scaler.fit_transform(X)

print("Features shape (scaled version):", X_sp.shape)

Features shape (scaled version): (96000, 7)


In [5]:
#apply PCA
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_sp)#Apply PCA
#print("PCA features shape:", X_pca.shape)
print("PCA-reduced features shape:", X_pca.shape)
print("Explained variance ratio sum:", sum(pca.explained_variance_ratio_))

PCA-reduced features shape: (96000, 6)
Explained variance ratio sum: 0.9563496543214839


In [6]:
#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [9]:
#K-Means on Scaled + PCA Data
start_time = time.time()
kmean_pca = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    kmean_pca.append({"algorithm": "KMeans", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")   


Runtime: 1043.2720320224762 seconds
K-Means runtime: 1043.2720 seconds


In [10]:
#Gaussian Mixture (GMM)on Scaled + PCA Data
start_time = time.time()
gmm_pca = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_pca)
    sil, db, ch = compute_metrics(X_pca, labels)
    gmm_pca.append({"algorithm": "GMM", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")   

Runtime: 1253.6933884620667 seconds
GMM runtime: 1253.6934 seconds


In [14]:
# from sklearn.preprocessing import MaxAbsScaler
# scaler = MaxAbsScaler()
# X_max_sp = scaler.fit_transform(X)

# print("Features shape (scaled version):", X_max_sp.shape)

# #apply PCA
# pca = PCA(n_components=0.95, random_state=42)
# X_pca = pca.fit_transform(X_max_sp)#Apply PCA
# #print("PCA features shape:", X_pca.shape)
# print("PCA-reduced features shape:", X_pca.shape)
# X_new = X_pca.astype("float32")
# start_time = time.time()

# agg_pca = []
# for k in k_values:
#     agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
#     labels = agg.fit_predict(X_new)
#     sil, db, ch = compute_metrics(X_new, labels)
#     agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

# end_time = time.time()
# runtime = end_time - start_time
# print("Runtime:", runtime, "seconds")
# print(f"Agglomerative runtime: {runtime:.4f} seconds")   

In [5]:
#Agglomerative Clustering on Scaled + PCA Data
subsample = np.random.choice(len(X_pca), 10000, replace=False)
X_sub = X_pca[subsample]
start_time = time.time()

agg_pca = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_sub)
    sil, db, ch = compute_metrics(X_sub, labels)
    agg_pca.append({"algorithm": "Agglomerative", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")   

Runtime: 44.15281295776367 seconds
Agglomerative runtime: 44.1528 seconds


In [7]:
#Spectral Clustering on Scaled + PCA Data
start_time = time.time()
subsample = np.random.choice(len(X_pca), 10000, replace=False)
X_sub = X_pca[subsample]
spec_pca = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_sub)
    sil, db, ch = compute_metrics(X_sub, labels)
    spec_pca.append({"algorithm": "Spectral", "preprocessing": "PCA", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 189.2372019290924 seconds
Spectral runtime: 189.2372 seconds


In [7]:
#DBSCAN on Scaled + PCA Data
start_time = time.time()
dbscan_pca = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_pca)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_pca[mask], labels[mask])
        dbscan_pca.append({"algorithm": "DBSCAN", "preprocessing": "PCA", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds")

Runtime: 316.6970989704132 seconds
DBSCAN runtime: 316.6971 seconds


In [11]:
#OPTICS on Scaled + PCA Data
start_time = time.time()
optics_pca = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_pca)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_pca, labels)
        optics_pca.append({
            "algorithm": "OPTICS",
            "preprocessing": "PCA",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"OPTICS runtime: {runtime:.4f} seconds")

Runtime: 12129.6520113945 seconds
OPTICS runtime: 12129.6520 seconds


In [15]:
#BIRCH on Scaled + PCA Data
start_time = time.time()
subsample = np.random.choice(len(X_pca), 10000, replace=False)
X_sub = X_pca[subsample]
birch_pca = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=8, threshold=t)
    labels = birch.fit_predict(X_sub)

    n_clusters = len(set(labels))
    if 1 < n_clusters < len(X_sub):
        sil, db, ch = compute_metrics(X_sub, labels)
        birch_pca.append({
            "algorithm": "BIRCH",
            "preprocessing": "PCA",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 12.482659101486206 seconds
BIRCH runtime: 12.4827 seconds


In [ ]:
import csv

pain_results_pca = (kmean_pca + gmm_pca + agg_pca + spec_pca + dbscan_pca + birch_pca + optics_pca)

keys = ["algorithm", "preprocessing", "k", "silhouette", "davies_bouldin", "calinski_harabasz", "eps"]

with open('updated_data/pain_data/pain_pca.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(pain_results_pca)
